# Earth2Studio CuPy DataArrays

Earth2Studio uses a normal `xarray.DataArray` as its shared data object. Xarray provides ordered dimensions, coordinates, and metadata, while the backing `data` is a NumPy array on CPU or a CuPy array on CUDA.

The `.e2s` accessor adds Earth2Studio operations such as CPU/GPU conversion, Torch conversion, batching, and grid metadata. This tutorial creates a small GPU-backed field, shares it with Torch, and then uses the same DataArray interface to declare model coordinate contracts without allocating field data.

In [ ]:
import numpy as np
import xarray as xr

import earth2studio as e2s
from earth2studio.models.px.fcn import VARIABLES as FCN_VARIABLES
from earth2studio.utils import from_torch

## A labeled GPU array

Start with the familiar xarray constructor. Calling `.e2s.as_cupy()` moves only the data payload to the GPU; its ordered dimensions, coordinates, name, and attributes remain attached.

In [ ]:
state = xr.DataArray(
    np.arange(8, dtype=np.float32).reshape(1, 2, 2, 2),
    dims=("batch", "variable", "lat", "lon"),
    coords={
        "variable": ["u10m", "v10m"],
        "lat": [40.0, 39.75],
        "lon": [250.0, 250.25],
    },
    name="wind",
).e2s.as_cupy()

{
    "dims": state.dims,
    "variables": state.coords["variable"].values,
    "gpu_backed": state.e2s.is_cupy,
}

## Share the payload with Torch

Torch models can consume the CuPy payload through DLPack without copying it. The temporary coordinate dictionary keeps legacy models working during the DataArray migration.

In [ ]:
tensor, coords = state.e2s.to_torch()
restored = from_torch(tensor, coords, name=state.name)
{
    "shared_gpu_memory": tensor.data_ptr() == state.data.data.ptr,
    "restored_dims": restored.dims,
}

## Batch dimensions for a model

Batching flattens selected dimensions while retaining enough coordinate information to restore them. Here the leading dimensions are contiguous, so both operations are views.

In [ ]:
packed = state.e2s.batch(
    ("batch", "variable"), batch_dim="sample", contiguous=False
)
unpacked = packed.e2s.unbatch(contiguous=False)
{
    "packed_shape": packed.shape,
    "restored_dims": unpacked.dims,
}

## Coordinate-only model contracts

Runtime DataArrays contain field values, but models also need to publish their input and output contracts before data exists. `coord_array()` returns a normal DataArray with a zero-byte shape-only payload. Fixed dimensions get their sizes from coordinates or grid metadata, while dynamic dimensions use length zero as a wildcard.

In [ ]:
e2s.known_grids(), e2s.resolve_grid("fcn")

### FourCastNet coordinate contract

The dimension tuple is the exact tensor order expected by the model. The `fcn` grid supplies latitude and longitude sizes, `variable` supplies channel labels, and the dynamic `batch` dimension accepts any runtime size.

In [ ]:
class ExampleFCN:
    step = np.timedelta64(6, "h")

    def input_coords(self):
        return e2s.coord_array(
            dims=("batch", "lead_time", "variable", "lat", "lon"),
            coords={"lead_time": [np.timedelta64(0, "h")], "variable": FCN_VARIABLES},
            dynamic=("batch",), grid="fcn",
        )

    def output_coords(self, coords):
        return coords.assign_coords(lead_time=coords.lead_time + self.step)

In [ ]:
model = ExampleFCN()
inputs = model.input_coords()
inputs.dims, inputs.shape, inputs.data.nbytes, inputs.e2s.get_grid()

## Materialize physical coordinates

A known grid can describe its geometry without storing large latitude and longitude arrays. Materialize them only when an operation needs physical coordinates; the field payload remains allocation-free.

In [ ]:
populated = inputs.e2s.materialize_grid_coords()
{
    "lat": populated.lat[[0, -1]].values,
    "lon": populated.lon[[0, -1]].values,
    "field_bytes": populated.data.nbytes,
}

## Derive an output contract

Output contracts are regular xarray transformations. One forecast step advances the lead-time coordinate while preserving dimension order, grid metadata, and the zero-byte payload.

In [ ]:
outputs = model.output_coords(inputs)
outputs.dims, outputs.lead_time.values, outputs.data.nbytes

## Use explicit coordinates

When coordinate values are already available, attach them using the normal xarray `coords` mapping. Earth2Studio validates that their sizes agree with the selected grid.

In [ ]:
explicit = e2s.coord_array(
    dims=inputs.dims, dynamic=("batch",), grid="fcn",
    coords={
        "lead_time": [np.timedelta64(0, "h")], "variable": FCN_VARIABLES,
        "lat": np.arange(90, -90, -0.25), "lon": np.arange(0, 360, 0.25),
    },
)
tuple(explicit.coords)

## Use projected and indexed grids

The same contract supports different spatial layouts. HRRR uses two projected dimensions with auxiliary two-dimensional latitude and longitude coordinates; HEALPix uses one pixel-index dimension with one-dimensional latitude and longitude coordinates.

In [ ]:
def grid_coords(grid, spatial_dims):
    return e2s.coord_array(
        dims=("batch", "variable", *spatial_dims),
        coords={"variable": ["u10m"]}, dynamic=("batch",), grid=grid,
    ).e2s.materialize_grid_coords()

hrrr = grid_coords("hrrr", ("hrrr_y", "hrrr_x"))
hpx = grid_coords("hpx6", ("hpx",))
[(a.e2s.get_grid(), tuple(a.coords), a.data.nbytes) for a in (hrrr, hpx)]

## Putting the pieces together

Runtime data and model contracts now use the same object. Data sources return populated NumPy- or CuPy-backed DataArrays, model signatures describe the required dimensions and metadata, and adapters convert to Torch only at legacy model boundaries.